In [ ]:


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib as mpl

# ── Force bold everywhere ──────────────────────────────────────────────────────
mpl.rcParams["font.weight"]        = "bold"
mpl.rcParams["axes.labelweight"]   = "bold"
mpl.rcParams["axes.titleweight"]   = "bold"
mpl.rcParams["figure.titleweight"] = "bold"

# ── Data ──────────────────────────────────────────────────────────────────────
cms = [
    np.array([[995,   4,   1],
              [ 10, 984,   6],
              [  3,   7, 990]]),   
    np.array([[987,  13,   0],
              [  8, 992,   0],
              [  3,   1, 996]]),   
    np.array([[996,   4,   0],
              [ 13, 987,   0],
              [  0,   0,1000]]),   
]

class_names  = ["N — Normal", "PB — Pre-Borderline", "PV — Post-Vascular"]
fold_labels  = ["Fold 1", "Fold 2", "Fold 3"]
n_classes, n_folds = 3, 3

def ovr_metrics(cm, ci):
    tp = cm[ci, ci]
    fn = cm[ci, :].sum() - tp
    fp = cm[:, ci].sum() - tp
    tn = cm.sum() - tp - fn - fp
    tpr  = tp / (tp + fn) if (tp + fn) else 0.
    fpr  = fp / (fp + tn) if (fp + tn) else 0.
    prec = tp / (tp + fp) if (tp + fp) else 0.
    f1   = 2*prec*tpr / (prec+tpr) if (prec+tpr) else 0.
    spec = tn / (tn + fp) if (tn + fp) else 0.
    return dict(tpr=tpr, fpr=fpr, prec=prec, f1=f1, spec=spec,
                tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))

results = [[ovr_metrics(cms[fi], ci) for fi in range(n_folds)]
           for ci in range(n_classes)]

def pseudo_curve(fpr_pt, tpr_pt, n=200):
    t1 = np.linspace(0, fpr_pt, n//2)
    y1 = np.interp(t1, [0, fpr_pt], [0, tpr_pt])
    t2 = np.linspace(fpr_pt, 1, n//2)
    y2 = np.interp(t2, [fpr_pt, 1], [tpr_pt, 1])
    return np.concatenate([t1, t2]), np.concatenate([y1, y2])

def auc_approx(fpr_pt, tpr_pt):
    x, y = pseudo_curve(fpr_pt, tpr_pt)
    return np.trapezoid(y, x)

# ── Theme ─────────────────────────────────────────────────────────────────────
BG      = "#ffffff"
PANEL   = "#f8fafc"
GRID    = "#e2e8f0"
TEXT    = "#1e293b"
MUTED   = "#475569"
ACCENT  = "#f1f5f9"

CLASS_COLORS = ["#0369a1", "#c2410c", "#15803d"]
FOLD_STYLES  = ["-", "--", "-."]
FOLD_COLORS  = ["#2563eb", "#db2777", "#d97706"]
FOLD_MARKERS = ["o", "s", "^"]

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 14), facecolor=BG)
gs  = GridSpec(n_classes, n_folds + 1, figure=fig,
               wspace=0.38, hspace=0.52,
               left=0.07, right=0.97, top=0.91, bottom=0.09)

axes = np.empty((n_classes, n_folds + 1), dtype=object)
for r in range(n_classes):
    for c in range(n_folds + 1):
        axes[r, c] = fig.add_subplot(gs[r, c])

def style_ax(ax, title="", xlabel=True, ylabel=True):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_color(GRID)
        sp.set_linewidth(1.2)
    ax.tick_params(colors=MUTED, labelsize=9, width=1.2)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight("bold")
    ax.grid(True, color=GRID, linewidth=0.8, alpha=0.9)
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    if xlabel:
        ax.set_xlabel("False Positive Rate (FPR)", color=MUTED,
                      fontsize=9, fontweight="bold", labelpad=5)
    if ylabel:
        ax.set_ylabel("True Positive Rate (TPR)", color=MUTED,
                      fontsize=9, fontweight="bold", labelpad=5)
    if title:
        ax.set_title(title, color=TEXT, fontsize=11,
                     fontweight="bold", pad=7)

# ── Plot ──────────────────────────────────────────────────────────────────────
for ci in range(n_classes):
    cls_col = CLASS_COLORS[ci]

    for fi in range(n_folds):
        ax  = axes[ci, fi]
        res = results[ci][fi]
        fpr_pt, tpr_pt = res["fpr"], res["tpr"]
        auc = auc_approx(fpr_pt, tpr_pt)

        style_ax(ax,
                 title=fold_labels[fi] if ci == 0 else "",
                 xlabel=(ci == n_classes - 1),
                 ylabel=(fi == 0))

        if fi == 0:
            ax.set_ylabel(class_names[ci], color=cls_col,
                          fontsize=10, fontweight="bold", labelpad=7)

        ax.plot([0, 1], [0, 1], "--", color=MUTED,
                linewidth=1.1, alpha=0.5)

        xs, ys = pseudo_curve(fpr_pt, tpr_pt)
        ax.plot(xs, ys, color=cls_col, linewidth=2.5,
                linestyle=FOLD_STYLES[fi], alpha=0.95)

        ax.scatter(fpr_pt, tpr_pt, color=cls_col,
                   marker=FOLD_MARKERS[fi], s=110,
                   edgecolors="white", linewidths=1.0, zorder=6)

        ax.plot([fpr_pt, fpr_pt], [0, tpr_pt], ":",
                color=cls_col, linewidth=1.0, alpha=0.5)
        ax.plot([0, fpr_pt], [tpr_pt, tpr_pt], ":",
                color=cls_col, linewidth=1.0, alpha=0.5)

        ax.text(0.97, 0.05, f"AUC \u2248 {auc:.4f}",
                transform=ax.transAxes, ha="right", va="bottom",
                color=cls_col, fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.35",
                          facecolor=ACCENT, edgecolor=cls_col,
                          linewidth=1.5, alpha=0.9))

        info = (f"TPR = {tpr_pt:.4f}\n"
                f"FPR = {fpr_pt:.4f}\n"
                f"F1  = {res['f1']:.4f}\n"
                f"Pre = {res['prec']:.4f}")
        ax.text(0.03, 0.97, info, transform=ax.transAxes,
                ha="left", va="top", color=TEXT,
                fontsize=8, fontweight="bold", linespacing=1.6,
                fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.4",
                          facecolor=ACCENT, edgecolor=GRID,
                          linewidth=1.2, alpha=0.85))

    # ── Summary col ───────────────────────────────────────────────────────────
    ax_sum = axes[ci, n_folds]
    style_ax(ax_sum,
             title="All Folds Overlay" if ci == 0 else "",
             xlabel=(ci == n_classes - 1),
             ylabel=False)

    ax_sum.plot([0, 1], [0, 1], "--", color=MUTED,
                linewidth=1.1, alpha=0.5)

    all_fpr = [results[ci][fi]["fpr"] for fi in range(n_folds)]
    all_tpr = [results[ci][fi]["tpr"] for fi in range(n_folds)]
    mean_fpr = np.mean(all_fpr); mean_tpr = np.mean(all_tpr)
    std_fpr  = np.std(all_fpr);  std_tpr  = np.std(all_tpr)
    mean_auc = auc_approx(mean_fpr, mean_tpr)

    for fi in range(n_folds):
        fpr_pt = results[ci][fi]["fpr"]
        tpr_pt = results[ci][fi]["tpr"]
        xs, ys = pseudo_curve(fpr_pt, tpr_pt)
        ax_sum.plot(xs, ys, color=FOLD_COLORS[fi],
                    linestyle=FOLD_STYLES[fi], linewidth=1.8,
                    alpha=0.75, label=fold_labels[fi])
        ax_sum.scatter(fpr_pt, tpr_pt,
                       color=FOLD_COLORS[fi], marker=FOLD_MARKERS[fi],
                       s=80, edgecolors="white", linewidths=0.8, zorder=6)

    xs_m, ys_m = pseudo_curve(mean_fpr, mean_tpr)
    ax_sum.plot(xs_m, ys_m, color=TEXT, linewidth=3.0,
                linestyle="-", alpha=0.92, label="Mean", zorder=5)
    ax_sum.scatter(mean_fpr, mean_tpr, color=TEXT,
                   marker="*", s=220, zorder=7,
                   edgecolors=cls_col, linewidths=1.2)
    ax_sum.errorbar(mean_fpr, mean_tpr,
                    xerr=std_fpr, yerr=std_tpr,
                    fmt="none", color=TEXT, alpha=0.35,
                    capsize=6, linewidth=1.8)

    ax_sum.text(0.97, 0.05, f"Mean AUC \u2248 {mean_auc:.4f}",
                transform=ax_sum.transAxes, ha="right", va="bottom",
                color=TEXT, fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.35",
                          facecolor=ACCENT, edgecolor=TEXT,
                          linewidth=1.5, alpha=0.9))

    if ci == 0:
        ax_sum.legend(loc="upper left",
                      framealpha=0.6, facecolor=PANEL,
                      edgecolor=GRID,
                      prop={"weight": "bold", "size": 8})

# ── Super-title ───────────────────────────────────────────────────────────────
fig.suptitle(
    "ROC Curve Comparison \u2014 All Folds \u00d7 All Classes  (One-vs-Rest Strategy)",
    color=TEXT, fontsize=15, fontweight="bold", y=0.975)

axes[0, n_folds].set_title("All Folds Overlay", color=TEXT,
                            fontsize=11, fontweight="bold", pad=7)

# ── Global legend ─────────────────────────────────────────────────────────────
legend_elems = [
    Line2D([0], [0], color=FOLD_COLORS[fi], linestyle=FOLD_STYLES[fi],
           linewidth=2.5, marker=FOLD_MARKERS[fi], markersize=7,
           label=fold_labels[fi],
           markerfacecolor=FOLD_COLORS[fi], markeredgecolor="white")
    for fi in range(n_folds)
] + [
    Line2D([0], [0], color=TEXT, linewidth=3.0, label="Mean curve"),
    Line2D([0], [0], color=MUTED, linewidth=1.2, linestyle="--",
           label="Random baseline"),
]
fig.legend(handles=legend_elems, loc="lower center", ncol=5,
           framealpha=0.6, facecolor=PANEL, edgecolor=GRID,
           fontsize=10, bbox_to_anchor=(0.5, 0.01),
           prop={"weight": "bold", "size": 10})

plt.savefig("roc_compare_folds.png", dpi=180,
            bbox_inches="tight", facecolor=BG)
print("Saved -> roc_compare_folds.png")
plt.show()